# [02 Feature Engineering & Temporal Splits]

Spec from `01_eda.ipynb` implemented. 

**Split** data into train/test sets first;
Then, apply **fit / transform pipeline**, to prevent leakage and mirror production.  

**`fit vs transform`:**
- `fit(train)` learns the stateful bits from **train only**: the `facid` category vocabulary, the glucose median, and the two winsorize caps.
- `transform(frame)` applies everything: stateless encodings + engineered features + one-hot (unknown category → `other`) + glucose impute + winsorize + drop.

> **Clean-code note (plan habit):** `fit_transformer` / `transform` are prototyped here, then lifted into `src/los_pred/features.py` and covered by `pytest`. Later they can graduate to a scikit-learn `Pipeline` (which serialises as one artifact for MLflow + the API).

## 0. Setup & load (raw)

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/LengthOfStay.csv")

print(df.shape)
df.head()


(100000, 28)


,eid,vdate,rcount,gender,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,psychother,fibrosisandother,malnutrition,hemo,hematocrit,neutrophils,sodium,glucose,bloodureanitro,creatinine,bmi,pulse,respiration,secondarydiagnosisnonicd9,discharged,facid,lengthofstay
0,1,8/29/2012,0,F,0,0,0,0,0,0,0,0,0,0,0,11.5,14.20,140.361132,192.476918,12.0,1.390722,30.432418,96,6.5,4,9/1/2012,B,3
1,2,5/26/2012,5+,F,0,0,0,0,0,0,0,0,0,0,0,9.0,4.10,136.731692,94.078507,8.0,0.943164,28.460516,61,6.5,1,6/2/2012,A,7
2,3,9/22/2012,1,F,0,0,0,0,0,0,0,0,0,0,0,8.4,8.90,133.058514,130.530524,12.0,1.065750,28.843812,64,6.5,2,9/25/2012,B,3
3,4,8/9/2012,0,F,0,0,0,0,0,0,0,0,0,0,0,11.9,9.40,138.994023,163.377028,12.0,0.906862,27.959007,76,6.5,1,8/10/2012,A,1
4,5,12/20/2012,0,F,0,0,0,1,0,1,0,0,0,0,0,9.1,9.05,138.634836,94.886654,11.5,1.242854,30.258927,67,5.6,2,12/24/2012,E,4


## 1 · Temporal batch label & split — BEFORE any transform

- `batch_month` = admission month, with the single-day `2013-01-01` stub **merged into 2012-12** (raw `vdate` left intact).
- Split by **time, not randomly**: earlier months → train, later → test (mirrors deployment: predict the future from the past).
- The split happens on **raw** data — the transform is fitted only on the resulting train slice.
- These monthly batches also serve as the simulated production batches for Evidently in Phase 1.

Default cutoff: train ≤ 2012-10, test 2012-11–12 (~83/17). Can be adjusted.

In [2]:
month = pd.to_datetime(df["vdate"]).dt.to_period("M")
df["batch_month"] = month.where(month < "2013-01", pd.Period("2012-12", freq="M"))

CUTOFF = pd.Period("2012-10", freq="M")   # last TRAIN month (inclusive)
is_train = df["batch_month"] <= CUTOFF
train_df, test_df = df[is_train].copy(), df[~is_train].copy()

print(f"train: {len(train_df):>6}  ({train_df['batch_month'].min()}..{train_df['batch_month'].max()})")
print(f"test:  {len(test_df):>6}  ({test_df['batch_month'].min()}..{test_df['batch_month'].max()})")

train:  83047  (2012-01..2012-10)
test:   16953  (2012-11..2012-12)


## 2. Define the transform (fit/transform)

**Stateless** (row-wise, no learned params): `isMale`, `rcount` `"5+"`→5, `comorbidity_count`.

**Stateful** (learned on train, applied everywhere):
- `facid` → one-hot over the **train** vocabulary, plus an explicit **`other`** column. Any category not seen in train (a new facility at serving time) maps to `other` → graceful fallback to the population baseline, instead of silently aliasing onto a real facility.
- `glucose` — impossible values (≤ 0) → **train** median. High glucose is clinically real, so it is **not** capped.
- `neutrophils`, `bloodureanitro` — generous upper cap via the **Tukey fence** `Q3 + 1.5·IQR` (learned on train). Loose on purpose: kill data-entry errors (~10× the 99th pct), not squash real highs. Swap to `99th·1.5` or a clinical bound by editing `fit_transformer`.

Dropped: `eid`, `vdate`, `discharged` (leakage), `gender`/`facid` (replaced), `respiration` (near-constant),`secondarydiagnosisnonicd9` (no target relationship), `batch_month`, and the target.

In [3]:
FLAGS = [
    "dialysisrenalendstage", "asthma", "irondef", "pneum", "substancedependence",
    "psychologicaldisordermajor", "depress", "psychother", "fibrosisandother",
    "malnutrition", "hemo",
]
TARGET = "lengthofstay"
DROP = ["eid", "vdate", "discharged", "gender", "facid", "respiration", "secondarydiagnosisnonicd9", "batch_month", TARGET]


# def _tukey_upper(s):
#     """Upper Tukey fence: Q3 + 1.5*IQR (generous outlier cap)."""
#     q1, q3 = s.quantile([0.25, 0.75])
#     return q3 + 2 * (q3 - q1)

def _cap_99th(s):
    return s.quantile(0.99)  # Cap at 99th percentile


def fit_transformer(train):
    """Learn all stateful params from TRAIN only."""
    return {
        "facid_categories": sorted(train["facid"].unique()),
        "glucose_median": train.loc[train["glucose"] > 0, "glucose"].median(),
        "neutrophils_cap": _cap_99th(train["neutrophils"]),
        "bloodureanitro_cap": _cap_99th(train["bloodureanitro"]),
    }


def transform(frame, p):
    """Apply the fitted transform. Pure: returns the feature matrix X (no target)."""
    out = frame.copy()

    # --- stateless encodings / engineered features ---
    out["isMale"] = (out["gender"] == "M").astype(int)
    out["rcount"] = pd.to_numeric(out["rcount"].replace("5+", "5"))
    out["comorbidity_count"] = out[FLAGS].sum(axis=1)

    # --- stateful: DQ fixes (params learned on train) ---
    out.loc[out["glucose"] <= 0, "glucose"] = p["glucose_median"]
    out["neutrophils"] = out["neutrophils"].clip(upper=p["neutrophils_cap"])
    out["bloodureanitro"] = out["bloodureanitro"].clip(upper=p["bloodureanitro_cap"])

    # --- stateful: facid one-hot over the TRAIN vocabulary + 'other' (OOV) ---
    cats = list(p["facid_categories"]) + ["other"]
    known = out["facid"].where(out["facid"].isin(p["facid_categories"]), "other")
    dummies = pd.get_dummies(pd.Categorical(known, categories=cats), prefix="fac", dtype=int)
    dummies.index = out.index
    out = pd.concat([out, dummies], axis=1)

    return out.drop(columns=[c for c in DROP if c in out.columns])

## 3 · Fit on train, apply to both

Because `facid` is one-hot over a fixed vocabulary (incl. `other`), train and test get an **identical column schema by construction** — no post-hoc `reindex` patch needed.

In [4]:
params = fit_transformer(train_df)
print("learned params:")
for k, v in params.items():
    print(f"  {k}: {v}")

X_train, y_train = transform(train_df, params), train_df[TARGET]
X_test, y_test = transform(test_df, params), test_df[TARGET]
print(f"\nX_train {X_train.shape}   X_test {X_test.shape}")
list(X_train.columns)

learned params:
  facid_categories: ['A', 'B', 'C', 'D', 'E']
  glucose_median: 142.1653124
  neutrophils_cap: 24.93333333
  bloodureanitro_cap: 53.0

X_train (83047, 28)   X_test (16953, 28)


['rcount',
 'dialysisrenalendstage',
 'asthma',
 'irondef',
 'pneum',
 'substancedependence',
 'psychologicaldisordermajor',
 'depress',
 'psychother',
 'fibrosisandother',
 'malnutrition',
 'hemo',
 'hematocrit',
 'neutrophils',
 'sodium',
 'glucose',
 'bloodureanitro',
 'creatinine',
 'bmi',
 'pulse',
 'isMale',
 'comorbidity_count',
 'fac_A',
 'fac_B',
 'fac_C',
 'fac_D',
 'fac_E',
 'fac_other']

## 4 · Sanity checks (prepare for Phase-1 pytest tests)

Run here first, then lift into `tests/test_features.py`.

In [5]:
# 1. No leakage / raw / target columns leaked into the feature matrix.
for col in ["eid", "vdate", "discharged", "gender", "facid", "respiration", "secondarydiagnosisnonicd9", TARGET]:
    assert col not in X_train.columns, col

# 2. Engineered feature in range, no NaN anywhere.
assert X_train["comorbidity_count"].between(0, 11).all()
assert not X_train.isna().any().any()

# 3. rcount fully numeric after the "5+" mapping.
assert pd.api.types.is_numeric_dtype(X_train["rcount"])

# 4. Train and test share an identical schema (order included).
assert list(X_train.columns) == list(X_test.columns)

# 5. glucose lows fixed; capped col does not exceed its learned cap.
assert (X_train["glucose"] > 0).all()
assert X_train["neutrophils"].max() <= params["neutrophils_cap"]

print("all sanity checks passed")

all sanity checks passed


In [6]:
# Unknown-category (OOV) behaviour: an unseen facility must route to fac_other.
demo = test_df.head(1).copy()
demo["facid"] = "Z"   # a facility never seen in train
tdemo = transform(demo, params)
known_fac_cols = [c for c in tdemo.columns if c.startswith("fac_") and c != "fac_other"]
assert tdemo["fac_other"].iloc[0] == 1
assert tdemo[known_fac_cols].iloc[0].sum() == 0
print("unknown facility 'Z' -> fac_other = 1, all known fac_* = 0  OK")

unknown facility 'Z' -> fac_other = 1, all known fac_* = 0  OK


## 5 · Save processed splits

Finally, save the split data so that Phase 2 (`train.py`) and Phase 1 monitoring reuse them without re-running this notebook. (Decide whether `data/processed/` is versioned in git, committed, or left for **DVC** later — the plan flags data versioning.)

In [7]:
from pathlib import Path

out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

X_train.assign(**{TARGET: y_train}).to_parquet(out / "train.parquet")
X_test.assign(**{TARGET: y_test}).to_parquet(out / "test.parquet")
print("saved:", sorted(p.name for p in out.glob('*.parquet')))



saved: ['test.parquet', 'train.parquet']


## 6 · Notes / decisions log

_Fill in as you run: chosen CUTOFF and split sizes; learned params (glucose median, the two Tukey caps); final feature count/list; anything surprising. Then move `fit_transformer` / `transform` into `src/los_pred/features.py` and write the pytest tests (including the OOV `fac_other` case)._

- **CUTOFF / split sizes:** 
>> CUTOFF at 2012-10; the train/test split:  83047/16953

- **Learned params:** 
>> `facid` → one-hot over the **train** vocabulary, plus an explicit **`other`** column. 
>> `glucose` — impossible values (≤ 0) → **train** median. 
>> `neutrophils`, `bloodureanitro`

- **Feature count & list:** 
>> Final feature list -
>> * ['rcount', 'dialysisrenalendstage','asthma', 'irondef', 'pneum','substancedependence', 'psychologicaldisordermajor', 'depress', 'psychother', 'fibrosisandother', 'malnutrition', 'hemo', 'hematocrit', 'neutrophils', 'sodium', 'glucose', 'bloodureanitro', 'creatinine', 'bmi', 'pulse', 'isMale', 'comorbidity_count', 'fac_A', 'fac_B', 'fac_C', 'fac_D', 'fac_E', 'fac_other']

- **Open questions:** 

>> If using XGBoost/LightGBM, replace clipping with NaN and remove this step